# Modelos Lineares

Um modelo linear combina as variáveis de entrada em uma soma ponderada acrescida de uma constante. O resultado serve como previsão em problemas de regressão e, restringido ao intervalo entre 0 e 1, como probabilidade em problemas de classificação. A mesma estrutura reaparece em cada camada de uma rede neural.

O notebook resolve a regressão linear pela equação normal e pelo gradiente descendente, compara as duas soluções, e adapta o modelo para classificação binária com a regressão logística.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

In [ ]:
# Fixa a semente do gerador de números aleatórios para tornar os resultados reproduzíveis
torch.manual_seed(42)

## Regressão linear

A regressão linear prevê um valor contínuo a partir das variáveis de entrada. Com um único atributo $x$, a previsão é a equação de uma reta,

$$
\hat{y} = w x + b
$$

em que $w$ é o peso, a inclinação da reta, e $b$ é o bias, o ponto onde ela cruza o eixo vertical. Com $n$ atributos, cada um recebe seu próprio peso e a previsão vira

$$
\hat{y} = \mathbf{w}^\top \mathbf{x} + b
$$

Treinar o modelo é encontrar $\mathbf{w}$ e $b$.

O dataset é sintético, gerado a partir de valores conhecidos de $w$ e $b$ com ruído gaussiano na saída. Conhecer os parâmetros de origem permite comparar, ao final, o que cada método recuperou.

In [ ]:
true_weight = 2.5
true_bias = 0.8

X = torch.randn(100, 1) * 2
noise = torch.randn(100, 1) * 1.5
y = true_weight * X + true_bias + noise

print(X.shape, y.shape)

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(X, y, s=20)
plt.xlabel("x")
plt.ylabel("y")
plt.grid(True)
plt.show()

### Erro quadrático médio

Comparar duas retas candidatas exige reduzir a qualidade de cada uma a um número. Para regressão, esse número é o erro quadrático médio,

$$
J(w, b) = \frac{1}{m} \sum_{i=1}^{m} \left( \hat{y}^{(i)} - y^{(i)} \right)^2
$$

O quadrado impede que erros de sinais opostos se cancelem e faz um erro grande pesar mais do que vários pequenos de mesma soma. Treinar é minimizar $J$.

In [ ]:
def mse(w, b):
    return ((w * X + b - y) ** 2).mean().item()


print(f"w=1.0 b=0.0: {mse(1.0, 0.0):.4f}")
print(f"w={true_weight} b={true_bias}: {mse(true_weight, true_bias):.4f}")

Com $b$ fixo, o custo em função de $w$ é uma parábola, com um único mínimo e sem mínimos locais. É essa forma que torna a minimização tratável.

In [ ]:
candidate_weights = torch.linspace(-2, 7, 100)
costs = [mse(w.item(), true_bias) for w in candidate_weights]

plt.figure(figsize=(8, 5))
plt.plot(candidate_weights, costs)
plt.axvline(true_weight, color="r", linestyle="--", label="w verdadeiro")
plt.xlabel("w")
plt.ylabel("J(w, b)")
plt.legend()
plt.grid(True)
plt.show()

### Equação normal

Igualar a zero a derivada do custo em relação aos parâmetros dá uma solução fechada, sem iteração. A matriz de entrada recebe uma coluna de uns, que multiplica o bias e o traz para dentro do mesmo vetor de parâmetros $\theta$:

$$
\hat{\theta} = (X_b^\top X_b)^{-1} X_b^\top y
$$

A inversão custa $O(n^3)$ no número de atributos e exige $X_b^\top X_b$ inversível. As duas condições inviabilizam o método para os milhões de parâmetros de uma rede neural.

In [ ]:
X_b = torch.cat([torch.ones(X.shape[0], 1), X], dim=1)
theta = torch.inverse(X_b.T @ X_b) @ X_b.T @ y

b_normal = theta[0].item()
w_normal = theta[1].item()
print(f"w={w_normal:.4f} b={b_normal:.4f}")

Os valores ficam próximos dos que geraram os dados. A diferença vem do ruído: o mínimo do custo sobre esta amostra não coincide com os parâmetros de origem.

In [ ]:
line_x = torch.linspace(X.min(), X.max(), 100).reshape(-1, 1)

plt.figure(figsize=(8, 5))
plt.scatter(X, y, s=20, label="dados")
plt.plot(line_x, w_normal * line_x + b_normal, color="r", label="equação normal")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.grid(True)
plt.show()

### Gradiente descendente

O gradiente $\nabla J(\theta)$ aponta na direção de maior crescimento do custo, então cada passo move os parâmetros na direção oposta,

$$
\theta \leftarrow \theta - \eta \nabla J(\theta)
$$

em que $\eta$ é a taxa de aprendizado. Valores pequenos tornam a convergência lenta, e valores grandes fazem os parâmetros ultrapassar o mínimo a cada passo, com o custo crescendo ao longo do treinamento.

O método vale para qualquer custo diferenciável e qualquer quantidade de parâmetros, e por isso é o algoritmo de treinamento das redes neurais.

O gradiente pode ser calculado sobre todo o conjunto de treinamento, sobre um único exemplo, ou sobre um lote pequeno, e as três variantes se chamam batch, stochastic e mini batch gradient descent. O `torch.optim.SGD` implementa a regra de atualização e não escolhe entre elas: quem escolhe é a quantidade de dados passada ao modelo por iteração. Aqui o dataset entra inteiro.

A camada `nn.Linear` guarda o peso e o bias como parâmetros e aplica $\mathbf{w}^\top \mathbf{x} + b$ à entrada. Os argumentos declaram quantos valores entram e quantos saem.

In [ ]:
model = nn.Linear(in_features=1, out_features=1)

print(model)
print(f"w inicial {model.weight.item():.4f}, b inicial {model.bias.item():.4f}")

Os valores iniciais são aleatórios e não guardam relação com os dados.

`nn.MSELoss` calcula o custo a partir das previsões e dos alvos, e `torch.optim.SGD` recebe os parâmetros do modelo e aplica a regra de atualização a cada chamada de `step`.

In [ ]:
learning_rate = 0.01
epochs = 200

loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

Cada época executa cinco operações: prever, calcular a perda, zerar os gradientes, propagar a derivada até os parâmetros e atualizá-los. O `zero_grad` é necessário porque `backward` soma os novos gradientes aos que já estão armazenados, em vez de substituí-los.

Esse ciclo é o mesmo para qualquer modelo em PyTorch.

In [ ]:
losses = []

for epoch in range(epochs):
    y_pred = model(X)                # forward pass
    loss = loss_fn(y_pred, y)        # perda da época

    optimizer.zero_grad()            # descarta os gradientes da época anterior
    loss.backward()                  # backward pass, calcula os gradientes
    optimizer.step()                 # atualiza os parâmetros

    losses.append(loss.item())
    if epoch % 20 == 0:
        print(f"época {epoch}: perda {loss.item():.4f}")

print(f"perda final: {losses[-1]:.4f}")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(losses)
plt.xlabel("época")
plt.ylabel("MSE")
plt.grid(True)
plt.show()

A curva cai rápido e estabiliza quando o gradiente encolhe. A tabela compara os parâmetros dos dois métodos com os que geraram os dados.

In [ ]:
pd.DataFrame(
    {
        "verdadeiro": [true_weight, true_bias],
        "equação normal": [w_normal, b_normal],
        "gradiente descendente": [model.weight.item(), model.bias.item()],
    },
    index=["w", "b"],
)

In [ ]:
with torch.no_grad():
    line_y = model(line_x)

plt.figure(figsize=(8, 5))
plt.scatter(X, y, s=20, label="dados")
plt.plot(line_x, w_normal * line_x + b_normal, color="r", label="equação normal")
plt.plot(line_x, line_y, color="g", linestyle="--", label="gradiente descendente")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.grid(True)
plt.show()

As duas retas se sobrepõem no gráfico. A diferença que resta na segunda casa do bias é o que faltaria percorrer com mais épocas. A vantagem do método iterativo é continuar aplicável quando a solução fechada deixa de existir.

## Regressão logística

A regressão logística resolve problemas de classificação binária. O modelo mantém a combinação linear das entradas e passa o resultado por uma função que o restringe ao intervalo entre 0 e 1, o que permite interpretá-lo como probabilidade.

### Função sigmoid

A sigmoid mapeia a reta real no intervalo aberto entre 0 e 1,

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

em que $z = \mathbf{w}^\top \mathbf{x} + b$ é a saída da parte linear. A saída se aproxima de 1 para $z$ muito positivo, de 0 para $z$ muito negativo, e vale exatamente 0.5 em $z = 0$. A classe atribuída é 1 quando $\sigma(z) \geq 0.5$.

In [ ]:
z = torch.linspace(-10, 10, 200)
sigmoid = 1 / (1 + torch.exp(-z))

plt.figure(figsize=(8, 5))
plt.plot(z, sigmoid)
plt.axhline(0.5, color="r", linestyle="--", label="limiar de decisão")
plt.xlabel("z")
plt.ylabel("sigma(z)")
plt.legend()
plt.grid(True)
plt.show()

O valor $z$ recebe o nome de logit, e o nome vem da função inversa da sigmoid,

$$
z = \log \frac{\hat{p}}{1 - \hat{p}}
$$

o logaritmo da razão entre a probabilidade da classe 1 e a da classe 0. O logit percorre toda a reta real, e a sigmoid o traz de volta ao intervalo das probabilidades. Em deep learning o termo designa qualquer saída de camada linear que ainda não passou pela função que a converte em probabilidade.

O dataset de classificação tem dois atributos e duas classes, com um agrupamento por classe. Dois atributos permitem representar cada exemplo como um ponto no plano e desenhar a fronteira aprendida.

In [ ]:
X_clf, y_clf = make_classification(
    n_samples=200,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    random_state=42,
)

X_clf = torch.from_numpy(X_clf).float()
y_clf = torch.from_numpy(y_clf).float().reshape(-1, 1)

print(X_clf.shape, y_clf.shape)

A divisão em treino e teste separa os exemplos que ajustam os parâmetros dos exemplos que medem o desempenho. Sem ela, a única métrica disponível seria calculada sobre dados que o modelo já viu.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)

In [ ]:
plt.figure(figsize=(8, 5))
for label in [0, 1]:
    mask = y_train.squeeze() == label
    plt.scatter(X_train[mask][:, 0], X_train[mask][:, 1], s=20, label=f"classe {label}")
plt.xlabel("x1")
plt.ylabel("x2")
plt.legend()
plt.grid(True)
plt.show()

### Entropia cruzada binária

O erro quadrático médio aplicado à saída da sigmoid produz um custo não convexo, com mínimos locais que travam o gradiente descendente. A perda usada em classificação binária é a entropia cruzada binária,

$$
J(\theta) = - \frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \log(\hat{p}^{(i)}) + (1 - y^{(i)}) \log(1 - \hat{p}^{(i)}) \right]
$$

Em cada exemplo apenas um dos dois termos sobrevive. Com $y = 1$ o custo é $-\log(\hat{p})$, próximo de zero quando $\hat{p}$ se aproxima de 1 e crescendo sem limite quando se aproxima de 0. A previsão confiante e errada é a mais penalizada.

O `nn.Sequential` aplica os módulos na ordem em que aparecem. A camada linear recebe dois atributos e produz um logit, que a sigmoid converte em probabilidade.

In [ ]:
classifier = nn.Sequential(
    nn.Linear(in_features=2, out_features=1),
    nn.Sigmoid(),
)

print(classifier)

Com a sigmoid dentro do modelo, a perda correspondente é `nn.BCELoss`, que recebe probabilidades. Em código de produção a escolha usual é o modelo devolver o logit e a perda ser `nn.BCEWithLogitsLoss`, que combina as duas etapas em um cálculo numericamente estável para logits extremos. A versão separada é usada aqui porque mantém visível a conversão em probabilidade.

In [ ]:
clf_learning_rate = 0.5
clf_epochs = 300

clf_loss_fn = nn.BCELoss()
clf_optimizer = torch.optim.SGD(classifier.parameters(), lr=clf_learning_rate)

O loop repete o ciclo da seção anterior com duas adições. A acurácia acompanha a perda, porque a perda não informa quantos exemplos o modelo acerta. E a cada época o modelo é avaliado também sobre o conjunto de teste, sem cálculo de gradientes, o que produz duas curvas comparáveis.

In [ ]:
train_losses = []
test_losses = []
train_accuracies = []
test_accuracies = []

for epoch in range(clf_epochs):
    y_prob = classifier(X_train)
    clf_loss = clf_loss_fn(y_prob, y_train)

    clf_optimizer.zero_grad()
    clf_loss.backward()
    clf_optimizer.step()

    with torch.no_grad():
        y_prob_test = classifier(X_test)
        clf_loss_test = clf_loss_fn(y_prob_test, y_test)

        train_losses.append(clf_loss.item())
        test_losses.append(clf_loss_test.item())
        train_accuracies.append((((y_prob >= 0.5).float() == y_train).float().mean().item()))
        test_accuracies.append((((y_prob_test >= 0.5).float() == y_test).float().mean().item()))

    if epoch % 50 == 0:
        print(f"época {epoch}: perda {clf_loss.item():.4f}, acurácia de teste {test_accuracies[-1]:.4f}")

print(f"acurácia final de teste: {test_accuracies[-1]:.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_losses, label="treino")
ax1.plot(test_losses, label="teste")
ax1.set_xlabel("época")
ax1.set_ylabel("entropia cruzada binária")
ax1.legend()
ax1.grid(True)

ax2.plot(train_accuracies, label="treino")
ax2.plot(test_accuracies, label="teste")
ax2.set_xlabel("época")
ax2.set_ylabel("acurácia")
ax2.legend()
ax2.grid(True)

plt.show()

A perda de teste alcança o menor valor por volta da época 35 e sobe um pouco depois, enquanto a acurácia de teste continua subindo. As duas métricas medem coisas diferentes: a perda leva em conta a probabilidade atribuída a cada exemplo, e a acurácia conta apenas de que lado do limiar ela caiu.

Os degraus vêm do tamanho do conjunto: com 40 exemplos de teste, um exemplo que muda de lado desloca a curva em 2.5 pontos percentuais.

### Fronteira de decisão

A fronteira de decisão é o conjunto de pontos em que o modelo atribui probabilidade 0.5 às duas classes. Como $\sigma(z) = 0.5$ apenas em $z = 0$, ela é dada por

$$
\mathbf{w}^\top \mathbf{x} + b = 0
$$

uma reta no plano dos dois atributos, e um hiperplano quando há mais atributos.

In [ ]:
x1 = torch.linspace(X_clf[:, 0].min() - 1, X_clf[:, 0].max() + 1, 300)
x2 = torch.linspace(X_clf[:, 1].min() - 1, X_clf[:, 1].max() + 1, 300)
grid_x1, grid_x2 = torch.meshgrid(x1, x2, indexing="xy")
grid = torch.stack([grid_x1.ravel(), grid_x2.ravel()], dim=1)

with torch.no_grad():
    grid_prob = classifier(grid).reshape(grid_x1.shape)

plt.figure(figsize=(8, 5))
plt.contourf(grid_x1, grid_x2, grid_prob, levels=[0, 0.5, 1], colors=["tab:blue", "tab:orange"], alpha=0.15)
for label in [0, 1]:
    mask = y_test.squeeze() == label
    plt.scatter(X_test[mask][:, 0], X_test[mask][:, 1], s=25, label=f"classe {label}")
plt.xlabel("x1")
plt.ylabel("x2")
plt.legend()
plt.grid(True)
plt.show()

A separação entre as regiões é uma reta, e essa é a única forma que o modelo produz. Um conjunto de dados cujas classes não se separam por uma reta está fora do que a regressão logística representa, por mais épocas de treinamento que receba.

## Exercícios

### Exercício 1

Gere um dataset sintético com três atributos, pesos verdadeiros à sua escolha e ruído gaussiano, e resolva a equação normal para recuperar os quatro parâmetros. Qual deles ficou mais distante do valor verdadeiro?

In [ ]:
true_weights = torch.tensor([[]])
true_intercept = 0.0

### Exercício 2

Treine um `nn.Linear(3, 1)` sobre o dataset do exercício anterior e compare os parâmetros aprendidos com os que a equação normal devolveu. Quantas épocas foram necessárias para a diferença entre os dois ficar abaixo de 0.01?

### Exercício 3

Treine o modelo de regressão linear da primeira parte com três taxas de aprendizado, uma muito menor que 0.01, uma próxima e uma muito maior, e desenhe as três curvas de perda no mesmo gráfico. A partir de qual valor a perda deixa de diminuir?

In [ ]:
learning_rates = []

### Exercício 4

Construa um dataset para a porta lógica AND e outro para a porta OR, com vários exemplos em torno de cada uma das quatro combinações de entrada e um pequeno ruído gaussiano, e treine um classificador para cada porta. Que acurácia cada um alcança?

In [ ]:
gate_inputs = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])

### Exercício 5

Repita o exercício anterior com a porta XOR, que vale 1 quando exatamente uma das entradas vale 1, e desenhe a fronteira de decisão obtida. Qual acurácia o modelo alcança e o que na fronteira explica esse limite?